<a href="https://colab.research.google.com/github/huseyin-karaca/s2t-tr-dev/blob/main/notebooks/colab/s2t_tr_dev_synthetic_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Synthetic Regime-Switch Experiment

End-to-end notebook for the synthetic experiment in Section 4.1 of the manuscript. The flow:

1. Clone the repo and set up the `uv` environment.
2. Generate the regime-switch parquet matching `src/data/dataset.py` schema.
3. Compute training-free baselines (random, weighted random, individual base models, oracle).
4. Train the proposed hierarchical transformer router.
5. Train the MLP-pool baseline router.
6. Evaluate both checkpoints on the same test split.
7. (Optional) Mirror the logs to Google Drive.

All cells use `!uv run python -m ...` so the notebook stays reproducible and matches the CLI used elsewhere in the project.

## 1. Initial setup (clone + uv env + secrets)

In [ ]:
! git clone https://github.com/huseyin-karaca/s2t-tr-dev
%cd /content/s2t-tr-dev

from google.colab import files
files.view('/content/s2t-tr-dev')

!make create_environment
!make requirements

from google.colab import userdata
import os
os.environ['GITHUB_TOKEN'] = userdata.get('GitHubPAT')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

!gh auth status
!git config --global user.name 'huseyin-karaca'
!git config --global user.email 'huseyinkaraccca@gmail.com'

Cloning into 's2t-tr-dev'...
remote: Enumerating objects: 347, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 347 (delta 44), reused 83 (delta 31), pack-reused 244 (from 2)
Receiving objects: 100% (347/347), 738.71 MiB | 19.93 MiB/s, done.
Resolving deltas: 100% (130/130), done.
Updating files: 100% (139/139), done.
/content/s2t-tr-dev


<IPython.core.display.Javascript object>

uv venv --python 3.10
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
>>> New uv virtual environment created. Activate with:
>>> Windows: .\.venv\Scripts\activate
>>> Unix/macOS: source ./.venv/bin/activate
uv sync
Resolved 199 packages in 23ms
Prepared 195 packages in 37.05s
Installed 195 packages in 309ms
 + absl-py==2.4.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + anyio==4.13.0
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==25.1.0
 + arrow==1.4.0
 + asttokens==3.0.1
 + async-lru==2.3.0
 + async-timeout==5.0.1
 + attrs==26.1.0
 + audioread==3.1.0
 + babel==2.18.0
 + beautifulsoup4==4.14.3
 + bleach==6.3.0
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + click==8.3.2
 + comm==0.2.3
 + contourpy==1.3.2
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.3
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + datasets==3.6.0
 + debugpy==1.8

## 2. Generate the synthetic regime-switch parquet

The output file follows the same schema as the real-world parquets: three frame-level feature columns and three per-expert WER columns. With the defaults (`N=10000`, `T=128`, `R=4`, `K=3`, `--feature-dtype float16`) the file is around 5 -- 6 GB on disk; the embeddings dominate. Generation is streamed in row groups of `--write-batch-size` clips, so peak RAM is bounded by one batch (~0.7 GB at default settings) regardless of `--num-samples`. Lower `--write-batch-size` if you have very tight RAM, or set `--feature-dtype float32` if you need full-precision embeddings (doubles disk and RAM).

In [ ]:
# Smoke run first — small file to confirm the pipeline before committing to the full size.
!uv run python -m src.data.synthetic \
    --output-path data/processed/synthetic_regime_switch_smoke/combined_features.parquet \
    --num-samples 200 --frame-length 32 --seed 42

In [ ]:
# Full synthetic dataset.
!uv run python -m src.data.synthetic \
    --output-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --num-samples 10000 --frame-length 128 \
    --num-regimes 4 --regime-dim 32 --noise-std 0.5 \
    --best-wer 0.15 --worst-wer 0.50 --wer-noise-std 0.02 \
    --seed 42

2026-04-26 16:14:13.430 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
2026-04-26 16:14:13,860 [INFO] __main__: Generating 10000 clips → data/processed/synthetic_regime_switch/combined_features.parquet (T=128, R=4, regime_dim=32, dims={'hubert': 1024, 'whisper': 512, 'wav2vec2': 1024}, dtype=float16)
2026-04-26 16:14:13,860 [INFO] __main__: Embedding payload: 6.55 GB total, peak per write batch: 0.33 GB (write_batch_size=500)
2026-04-26 16:15:01,769 [INFO] __main__:   wrote 2000 / 10000 clips
2026-04-26 16:15:48,024 [INFO] __main__:   wrote 4000 / 10000 clips
2026-04-26 16:16:34,273 [INFO] __main__:   wrote 6000 / 10000 clips
2026-04-26 16:17:21,414 [INFO] __main__:   wrote 8000 / 10000 clips
2026-04-26 16:18:08,756 [INFO] __main__:   wrote 10000 / 10000 clips
2026-04-26 16:18:08,758 [INFO] __main__: Sanity stats — random WER: 0.3785, oracle WER: 0.1531, per-expert mean WER: {'hubert': 0.43208104372024536, 'whisper': 0.344176709651947, 'wav2vec2': 0.3591602

## 3. Inspect the synthetic dataset structure

A four-panel sanity figure produced directly from the parquet metadata: regime centers in the latent space (PCA), the per-(r1, r2) best-expert grid, the per-expert WER table heatmaps, and a few per-clip frame trajectories. Use it to confirm that regimes are separable, the WER table is asymmetric in the ordered pair, and clips show a clean regime switch in their embeddings.

In [ ]:
!uv run python -m src.data.synthetic_inspect \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --output-path  reports/figures/synthetic_inspect.pdf \
    --num-trajectories 4 --seed 0

2026-04-26 16:18:09.039 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
2026-04-26 16:18:12,553 [INFO] __main__: Loaded synthetic metadata from data/processed/synthetic_regime_switch/combined_features.parquet — R=4, regime_dim=32, K=3
2026-04-26 16:18:25,251 [INFO] __main__: Wrote inspection figure to reports/figures/synthetic_inspect.pdf


## Run the Synthetic Experiment Pipeline

We use `main_results` to orchestrate training the hierarchical transformer and the MLP baseline using the `synthetic.yaml` config.
This will automatically run training-free baselines (if applicable) and train all variants defined in the config.

In [ ]:
!uv run python -m src.experiments.main_results experiments=synthetic

## Post-hoc visualizations of both trained checkpoints

Confusion matrices, per-clip WER scatters, per-base-model selection frequencies, and overlaid training curves.

In [ ]:
!uv run python -m src.training.visualize predictions \
    --parquet-path data/processed/synthetic_regime_switch/combined_features.parquet \
    --ckpt proposed=reports/main_results/synthetic/synthetic_regime_switch_hier/checkpoints/last.ckpt \
    --ckpt mlp_pool=reports/main_results/synthetic/synthetic_regime_switch_mlp_pool/checkpoints/last.ckpt \
    --output-dir reports/figures/synthetic_regime_switch \
    --split test --train-ratio 0.8 --val-ratio 0.1 --seed 42 \
    --max-seq-len 256 --batch-size 32 --num-workers 4

In [ ]:
!uv run python -m src.training.visualize curves \
    --logdir proposed=reports/main_results/synthetic/synthetic_regime_switch_hier \
    --logdir mlp_pool=reports/main_results/synthetic/synthetic_regime_switch_mlp_pool \
    --output-path reports/figures/synthetic_regime_switch/curves.pdf

## Sweep over the number of regimes

Re-runs the entire pipeline for each value of `R`.

In [ ]:
!uv run python -m src.experiments.synthetic_sweep run \
    --output-dir reports/sweeps/synthetic_R \
    --r-values 2,4,6,8 \
    --num-samples 5000 --frame-length 128 --regime-dim 32 --noise-std 0.5 \
    --max-seq-len 256 --batch-size 64 --num-workers 4 \
    --max-epochs 1 \
    --primary-weight 1.0 --aux-ce-weight 0.0 \
    --soft-ce-weight 0.5 --soft-ce-temperature 0.1 \
    --seed 42 --keep-parquets

In [ ]:
!uv run python -m src.experiments.synthetic_sweep figure \
    --results reports/sweeps/synthetic_R/sweep_results.json \
    --output-path reports/manuscript/figures/synthetic_sweep.pdf

## Pull Results from W&B

This script queries the W&B API for runs matching this experiment group and prints a paper-ready markdown table.

In [ ]:
# Ensure WANDB_API_KEY and WANDB_ENTITY are set in your Colab Secrets or environment
!uv run python -m src.scripts.pull_wandb \
    --api-key $WANDB_API_KEY \
    --entity $WANDB_ENTITY \
    --group synthetic